# Fed-XRay: Free Cloud GPU Runner (Google Colab / Kaggle)

Bu notebook, **Fed-XRay** projesinin FastAPI ve PyTorch arka plan sunucusunu **ücretsiz bulut GPU (NVIDIA T4 / 16-32 GB VRAM + 13-30 GB RAM)** üzerinde çalıştırır ve **Cloudflare Tunnel** ile güvenli genel bir HTTPS bağlantı adresi üretir.

### Adımlar:
1. **Menüden GPU Seçin:** `Runtime` -> `Change runtime type` -> `T4 GPU` seçin ve `Save` deyin.
2. **Tümünü Çalıştırın:** `Runtime` -> `Run all` (veya `Ctrl + F9`).
3. En alttaki hücrede çıkan `https://xxxx.trycloudflare.com` adresini kopyalayıp yerel frontend'inizde kullanın.

In [1]:
# 1. GPU ve Donanım Bilgisini Kontrol Edin
!nvidia-smi

Sat Oct 31 23:23:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Depoyu Klonlayın ve Bağımlılıkları Yükleyin
import os
if not os.path.exists("Fed-XRay"):
    !git clone https://github.com/yusufcalisir/Fed-XRay.git
%cd Fed-XRay
!git pull origin main
!pip install -r requirements.txt

Cloning into 'Fed-XRay'...
remote: Enumerating objects: 913, done.
remote: Counting objects: 100% (913/913), done.
remote: Compressing objects: 100% (525/525), done.
remote: Total 913 (delta 537), reused 707 (delta 342), pack-reused 0 (from 0)
Receiving objects: 100% (913/913), 478.03 KiB | 12.58 MiB/s, done.
Resolving deltas: 100% (537/537), done.
/content/Fed-XRay
From https://github.com/yusufcalisir/Fed-XRay
 * branch            main       -> FETCH_HEAD
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 21.4 MB/s  0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.5.0
    Uninstalling click-8.5.0:
      Successfully uninstalled click-8.5.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [gTTS]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which 

In [3]:
# 3. Cloudflare Tünelini ve FastAPI Sunucusunu Başlatın
import subprocess
import time
import re
import os

# Cloudflare Tunnel indir
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# Uvicorn backend başlat
print("Starting Fed-XRay FastAPI backend on port 8000...")
backend_proc = subprocess.Popen(
    ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
time.sleep(3)

# Cloudflare Tüneli başlat
print("Launching Cloudflare Secure Tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    if line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break
    time.sleep(0.5)

if public_url:
    print("\n" + "="*70)
    print(" FED-XRAY BULUT GPU SUNUCUSU AKTIF!")
    print(f" BAGLANTI ADRESINIZ (Public API URL): {public_url}")
    print("="*70)
    print("\nBu adresi kopyalayip bilgisayarinizdaki frontend/.env.local dosyasina ekleyin:")
    print(f"NEXT_PUBLIC_API_URL={public_url}")
else:
    print("Tunel baglantisi saglanamadi. Lutfen loglari kontrol edin.")

Starting Fed-XRay FastAPI backend on port 8000...
Launching Cloudflare Secure Tunnel...

 FED-XRAY BULUT GPU SUNUCUSU AKTIF!
 BAGLANTI ADRESINIZ (Public API URL): https://globe-celebration-poems-shakira.trycloudflare.com

Bu adresi kopyalayip bilgisayarinizdaki frontend/.env.local dosyasina ekleyin:
NEXT_PUBLIC_API_URL=https://globe-celebration-poems-shakira.trycloudflare.com


In [ ]:
# 4. Canlı Sunucu Loglarını Takip Edin (Oturumu Açık Tutar)
try:
    while True:
        line = backend_proc.stdout.readline()
        if line:
            print(line.strip())
        time.sleep(0.1)
except KeyboardInterrupt:
    print("Sunucu kapatiliyor...")
    backend_proc.terminate()
    tunnel_proc.terminate()

INFO:     Started server process [753]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     195.155.123.136:0 - "GET /api/health HTTP/1.1" 200 OK
INFO:     195.155.123.136:0 - "GET /api/state HTTP/1.1" 200 OK
INFO:     195.155.123.136:0 - "GET / HTTP/1.1" 200 OK
INFO:     195.155.123.136:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     195.155.123.136:0 - "GET /api/health HTTP/1.1" 200 OK
INFO:     195.155.123.136:0 - "GET /api/state HTTP/1.1" 200 OK
INFO:     195.155.123.136:0 - "OPTIONS /api/cohorts/generate HTTP/1.1" 200 OK
INFO:     195.155.123.136:0 - "POST /api/cohorts/generate HTTP/1.1" 200 OK
INFO:     195.155.123.136:0 - "GET /api/health HTTP/1.1" 200 OK
INFO:     195.155.123.136:0 - "GET /api/health HTTP/1.1" 200 OK
INFO:     195.155.123.136:0 - "GET /api/health HTTP/1.1" 200 OK
INFO:     195.155.123.136:0 - "GET /api/health HTTP/1.1" 200 OK
INFO:     195.155